# Week 4: Reinforcement Learning for Dynamic Pricing
## Multi-Armed Bandits + Q-Learning

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_style('whitegrid')
%matplotlib inline

np.random.seed(42)

## Part 1: Multi-Armed Bandit Problem

In [ ]:
# Environment setup
PRICES = [5, 10, 15, 20, 25]
TRUE_CONVERSION_RATES = [0.50, 0.35, 0.22, 0.12, 0.05]
N_ARMS = len(PRICES)

print("Price Options and True Conversion Rates:")
for price, rate in zip(PRICES, TRUE_CONVERSION_RATES):
    expected_revenue = price * rate
    print(f"  ${price}: {rate:.0%} conversion → ${expected_revenue:.2f} expected revenue")

# Best arm
expected_revenues = [p * r for p, r in zip(PRICES, TRUE_CONVERSION_RATES)]
best_arm = np.argmax(expected_revenues)
print(f"\nOptimal price: ${PRICES[best_arm]} (expected revenue: ${expected_revenues[best_arm]:.2f})")

In [ ]:
# Bandit agent
class MultiArmedBandit:
    def __init__(self, n_arms):
        self.n_arms = n_arms
        self.counts = np.zeros(n_arms)
        self.values = np.zeros(n_arms)
        self.cumulative_reward = []
        self.total_reward = 0
    
    def update(self, arm, reward):
        self.counts[arm] += 1
        n = self.counts[arm]
        self.values[arm] += (reward - self.values[arm]) / n
        self.total_reward += reward
        self.cumulative_reward.append(self.total_reward)

# Simulate customer purchase
def simulate_purchase(arm):
    price = PRICES[arm]
    conversion_rate = TRUE_CONVERSION_RATES[arm]
    converted = np.random.random() < conversion_rate
    return price if converted else 0

## Strategy 1: Random Selection

In [ ]:
def random_strategy(bandit):
    return np.random.randint(bandit.n_arms)

# Run simulation
n_customers = 1000
bandit_random = MultiArmedBandit(N_ARMS)

for t in range(n_customers):
    arm = random_strategy(bandit_random)
    reward = simulate_purchase(arm)
    bandit_random.update(arm, reward)

print("Random Strategy:")
print(f"Total revenue: ${bandit_random.total_reward:.2f}")
print(f"Average revenue per customer: ${bandit_random.total_reward/n_customers:.2f}")
print(f"\nPrices selected: {bandit_random.counts.astype(int)}")

## Strategy 2: Epsilon-Greedy

In [ ]:
def epsilon_greedy_strategy(bandit, epsilon):
    if np.random.random() < epsilon:
        return np.random.randint(bandit.n_arms)
    return int(np.argmax(bandit.values))

# Run simulation
epsilon = 0.1
bandit_egreedy = MultiArmedBandit(N_ARMS)

for t in range(n_customers):
    arm = epsilon_greedy_strategy(bandit_egreedy, epsilon)
    reward = simulate_purchase(arm)
    bandit_egreedy.update(arm, reward)

print(f"Epsilon-Greedy (ε={epsilon}):")
print(f"Total revenue: ${bandit_egreedy.total_reward:.2f}")
print(f"Average revenue per customer: ${bandit_egreedy.total_reward/n_customers:.2f}")
print(f"\nPrices selected: {bandit_egreedy.counts.astype(int)}")
print(f"Estimated values: {np.round(bandit_egreedy.values, 2)}")

## Strategy 3: Upper Confidence Bound (UCB)

In [ ]:
def ucb_strategy(bandit, t, confidence=2.0):
    # Initially try all arms
    for arm in range(bandit.n_arms):
        if bandit.counts[arm] == 0:
            return arm
    
    # Calculate UCB values
    ucb_values = bandit.values + confidence * np.sqrt(np.log(t + 1) / bandit.counts)
    return int(np.argmax(ucb_values))

# Run simulation
bandit_ucb = MultiArmedBandit(N_ARMS)

for t in range(n_customers):
    arm = ucb_strategy(bandit_ucb, t, confidence=2.0)
    reward = simulate_purchase(arm)
    bandit_ucb.update(arm, reward)

print("UCB Strategy:")
print(f"Total revenue: ${bandit_ucb.total_reward:.2f}")
print(f"Average revenue per customer: ${bandit_ucb.total_reward/n_customers:.2f}")
print(f"\nPrices selected: {bandit_ucb.counts.astype(int)}")
print(f"Estimated values: {np.round(bandit_ucb.values, 2)}")

## Compare All Strategies

In [ ]:
# Plot cumulative rewards
plt.figure(figsize=(12, 6))
plt.plot(bandit_random.cumulative_reward, label='Random', alpha=0.7)
plt.plot(bandit_egreedy.cumulative_reward, label='Epsilon-Greedy', alpha=0.7)
plt.plot(bandit_ucb.cumulative_reward, label='UCB', alpha=0.7)
plt.xlabel('Number of Customers')
plt.ylabel('Cumulative Revenue ($)')
plt.title('Bandit Strategy Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Summary
results_df = pd.DataFrame({
    'Strategy': ['Random', 'Epsilon-Greedy', 'UCB'],
    'Total Revenue': [
        bandit_random.total_reward,
        bandit_egreedy.total_reward,
        bandit_ucb.total_reward
    ],
    'Avg per Customer': [
        bandit_random.total_reward / n_customers,
        bandit_egreedy.total_reward / n_customers,
        bandit_ucb.total_reward / n_customers
    ]
})

print("\nStrategy Comparison:")
print(results_df.round(2))

## Part 2: Q-Learning for Dynamic Pricing

In [ ]:
# Environment with states
INVENTORY_LEVELS = ['high', 'low']
TIME_PERIODS = ['early', 'late']
STATES = [(inv, time) for inv in INVENTORY_LEVELS for time in TIME_PERIODS]
STATE_TO_IDX = {state: idx for idx, state in enumerate(STATES)}
N_STATES = len(STATES)
N_ACTIONS = len(PRICES)

print("States:")
for i, state in enumerate(STATES):
    print(f"  {i}: {state[0]} inventory, {state[1]} time")

In [ ]:
# Environment
class PricingEnvironment:
    def __init__(self):
        self.state_idx = 0
    
    def reset(self):
        self.state_idx = np.random.randint(N_STATES)
        return self.state_idx
    
    def get_conversion_rate(self, state_idx, action):
        base_rate = TRUE_CONVERSION_RATES[action]
        state = STATES[state_idx]
        inv, time = state
        
        # Modifiers based on state
        if inv == 'high' and time == 'early':
            modifier = 1.0
        elif inv == 'high' and time == 'late':
            modifier = 1.2  # Urgency
        elif inv == 'low' and time == 'early':
            modifier = 0.9  # Can be selective
        else:
            modifier = 1.1
        
        return min(1.0, base_rate * modifier)
    
    def step(self, action):
        price = PRICES[action]
        conversion_rate = self.get_conversion_rate(self.state_idx, action)
        converted = np.random.random() < conversion_rate
        reward = price if converted else 0
        
        # Random state transition
        next_state = np.random.randint(N_STATES)
        done = np.random.random() < 0.1
        
        self.state_idx = next_state
        return next_state, reward, done

In [ ]:
# Q-Learning Agent
class QLearningAgent:
    def __init__(self, n_states, n_actions, learning_rate=0.1, discount=0.9, epsilon=0.1):
        self.n_states = n_states
        self.n_actions = n_actions
        self.lr = learning_rate
        self.gamma = discount
        self.epsilon = epsilon
        self.q_table = np.zeros((n_states, n_actions))
    
    def choose_action(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        return int(np.argmax(self.q_table[state]))
    
    def update(self, state, action, reward, next_state):
        best_next = np.max(self.q_table[next_state])
        td_target = reward + self.gamma * best_next
        td_error = td_target - self.q_table[state, action]
        self.q_table[state, action] += self.lr * td_error
    
    def get_policy(self):
        policy = {}
        for state_idx, state in enumerate(STATES):
            best_action = int(np.argmax(self.q_table[state_idx]))
            policy[f"{state[0]}_{state[1]}"] = PRICES[best_action]
        return policy

## Train Q-Learning Agent

In [ ]:
# Training
episodes = 1000
env = PricingEnvironment()
agent = QLearningAgent(N_STATES, N_ACTIONS, learning_rate=0.1, discount=0.9, epsilon=0.1)

episode_rewards = []
rolling_avg = []

for episode in range(episodes):
    state = env.reset()
    episode_reward = 0
    done = False
    steps = 0
    max_steps = 100
    
    while not done and steps < max_steps:
        action = agent.choose_action(state)
        next_state, reward, done = env.step(action)
        agent.update(state, action, reward, next_state)
        episode_reward += reward
        state = next_state
        steps += 1
    
    episode_rewards.append(episode_reward)
    
    # Rolling average (last 100 episodes)
    if len(episode_rewards) >= 100:
        rolling_avg.append(np.mean(episode_rewards[-100:]))
    else:
        rolling_avg.append(np.mean(episode_rewards))

print("Training complete!")
print(f"Final rolling average reward: ${rolling_avg[-1]:.2f}")

## Learning Curve

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(rolling_avg, linewidth=2)
plt.xlabel('Episode')
plt.ylabel('Average Reward (100-episode window)')
plt.title('Q-Learning: Learning Curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Learned Policy

In [ ]:
# Display Q-table
q_df = pd.DataFrame(
    agent.q_table,
    index=[f"{s[0]}_{s[1]}" for s in STATES],
    columns=[f"${p}" for p in PRICES]
)

print("Q-Table:")
print(q_df.round(2))

# Display policy
policy = agent.get_policy()
print("\nLearned Policy (optimal price for each state):")
for state_name, price in policy.items():
    print(f"  {state_name}: ${price}")

## Visualize Q-Table as Heatmap

In [ ]:
plt.figure(figsize=(10, 5))
sns.heatmap(agent.q_table, annot=True, fmt='.2f', cmap='YlGnBu',
            xticklabels=[f"${p}" for p in PRICES],
            yticklabels=[f"{s[0]}_{s[1]}" for s in STATES],
            cbar_kws={'label': 'Q-Value'})
plt.xlabel('Action (Price)')
plt.ylabel('State')
plt.title('Q-Learning: Learned Q-Table')
plt.tight_layout()
plt.show()